# 🥉 **Ingesta de Datos - Capa Bronze**
*Datos crudos tal como vienen de la fuente, sin transformaciones.*

In [0]:
from databricks.connect import DatabricksSession
from pyspark.sql import DataFrame
from pyspark.sql.utils import AnalysisException
import logging

# --- Configuración de logging -------------------------------------------------
logging.basicConfig(level=logging.INFO, format = '\33[30m%(asctime)s [%(levelname)s]\33[0m %(message)s')
logger = logging.getLogger(__name__)

# --- Constantes --------------------------------------------------------------- 
BASE_PATH = '/Workspace/Users/raul.perez.costero@gmail.com/.bundle/Data Engineer/dev/files/databriks-repositorio/notebooks'

CATALOG = 'workspace'

SCHEMA = 'retail_db'

SOURCES = {
    "orders":   "olist_orders_dataset.csv",
    "items":    "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
} 

# --- Helpers -------------------------------------------------------------------
def read_csv(spark: DatabricksSession, path: str) -> DataFrame:
    '''Lee un CSV con cabecera y lo devuelve como un DataFrame'''
    return (
        spark.read
        .format('csv')
        .options(header = True, inferSchema=True)
        .load(path)
        )

def save_as_bronze(df: DataFrame, table: str) -> None:
    '''Escribe un DataFrame como un Delta Lake'''
    full_table = f'{CATALOG}.{SCHEMA}.{table}'
    (
        df.write
        .format('delta')
        .mode('overwrite')
        .options(overwriteSchema = True)
        .saveAsTable(full_table)
     )
    logging.info(f'\33[35mTabla {full_table} guardada correctamente.\33[0m')


# --- Pipeline principal ---------------------------------------------------------
def main() -> None:
    spark = DatabricksSession.builder.getOrCreate()

    # 1. Crear schema si no existe
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
    logger.info(f"\33[36mSchema '{CATALOG}.{SCHEMA}' verificado.\33[0m")

    # 2. Leer → escribir cada fuente
    for table_name, filename in SOURCES.items():
        path = f"{BASE_PATH}/{filename}"
        try:
            logger.info(f"\33[33mLeyendo {filename}...\33[0m")
            df = read_csv(spark, path)
            save_as_bronze(df, table_name)
        except AnalysisException as e:
            logger.error(f"\33[31mError procesando '{filename}': {e}\33[0m")
            raise

    logger.info(f"\33[32m✅ Tablas Bronze creadas correctamente en {CATALOG}.{SCHEMA}✅\33[0m")

if __name__ == "__main__":
    main()



In [0]:
# Establece el espacio de trabajo y la base de datos, para evitar tener que especificarlo en cada consulta.
# Evita: SELECT * FROM workspace.retail_db.items LIMIT 3
spark.catalog.setCurrentCatalog("workspace")
spark.catalog.setCurrentDatabase("retail_db")

In [0]:
# Consulta desde Python-SQL:
spark.sql('SELECT * FROM items LIMIT 3').show()

In [0]:
%sql
--Consulta desde SQL:
DESCRIBE items


# 🥈 **Transformación de Datos - Capa Silver**
*Datos limpios, validados y estructurados listos para analizar.*

# 🥇 **Modelado de Datos - Capa Gold**
*Datos agregados y optimizados para consumo de negocio.*

# 💎 **Serving Layer - Capa Platinum** 
*Métricas y KPIs finales expuestos para dashboards y reportes.*

In [0]:
## No tiene sentido hacer esta capa en un notebook, usar Power BI + Snowflake, MLflow, Databricks Model Serving, , Tableau, Looker